In [ ]:
import random
from functools import partial
import matplotlib.pyplot as plt
import numpy as np
import mph
from gefest.core.configs.optimization_params import OptimizationParams
from gefest.core.configs.tuner_params import TunerParams
from gefest.core.geometry.datastructs.structure import Structure
from gefest.core.geometry.domain import Domain
from gefest.core.opt.objective.objective import Objective
from gefest.tools.estimators.estimator import Estimator
from setup_comsol import simulate_hydrodynamics
from gefest.core.opt.analytics import EvoAnalytics
from gefest.tools.optimizers.optimizer import Optimizer

from gefest.core.geometry.domain import Domain
from gefest.core.geometry.datastructs.structure import Structure
from gefest.core.viz.struct_vizualizer import StructVizualizer
from gefest.core.viz.struct_vizualizer import GIFMaker
from tqdm import tqdm
import os
random.seed(42)
np.random.seed(42)

In [2]:
EvoAnalytics.clear()

In [3]:
comsol_objective = partial(simulate_hydrodynamics, client=mph.Client(cores=8))

2025-12-09 16:33:05,345 - JPype version is 1.6.0.
2025-12-09 16:33:05,346 - Starting Java virtual machine.
2025-12-09 16:33:07,480 - Java virtual machine has started.
2025-12-09 16:33:07,481 - Initializing stand-alone client.
2025-12-09 16:33:09,931 - Stand-alone client initialized.
2025-12-09 16:33:10,028 - Running on 8 processor cores.


In [ ]:
domain_cfg = Domain(allowed_area=[(15.0, 15.0),
                                  (15.0, 335.0),
                                  (215.0, 335.0),
                                  (140.0, 125.0),
                                  (130.0, 15.0),],
                max_poly_num=8,
                min_poly_num=1,
                )

In [5]:
tuner_cfg = TunerParams(
    tuner_type='optuna',
    n_steps_tune=10,
    hyperopt_dist='uniform',
    verbose=True,
    timeout_minutes=60,
)

In [6]:
opt_params = OptimizationParams(
    optimizer='gefest_ga',
    domain=domain_cfg,
    tuner_cfg=tuner_cfg,
    n_steps=60,
    pop_size=40,
    postprocess_attempts=10,
    mutation_prob=0.5,
    crossover_prob=0.5,
    mutations=[
        'rotate_poly',
        'resize_poly',
        # 'add_point',
        # 'drop_point',
        # 'add_poly',
        # 'drop_poly',
        'pos_change_point',
    ],
    selector='tournament_selection',
    # mutation_each_prob=[0.125, 0.125, 0.15, 0.35, 0.00, 0.00, 0.25],
    mutation_each_prob=[0.25, 0.25, 0.5],
    crossovers=[
        'polygon_level',
        'structure_level',
    ],
    crossover_each_prob=[0.7, 0.3],
    postprocess_rules=[
        'not_out_of_bounds',
        'valid_polygon_geom',
        'not_self_intersects',
        'not_too_close_polygons',
        'not_too_close_points',
    ],
    extra=5,
    n_jobs=1,
    log_dir='logs',
    run_name='run_name',
    golem_keep_histoy=False,
    golem_genetic_scheme_type='steady_state',
    golem_surrogate_each_n_gen=5,
    objectives=[
        comsol_objective,
    ],
)

In [ ]:
optimizer = opt_params.optimizer(opt_params)

In [ ]:
optimized_pop = optimizer.optimize()

In [16]:
import pickle
best = min(optimized_pop, key=lambda s: s.fitness)

with open('best_structure.pkl', 'wb') as f:
    pickle.dump(best, f)

In [3]:
import pickle
with open('best_structure.pkl', 'rb') as f:
    best_val = pickle.load(f)
client = mph.Client(cores=1)

2025-12-08 15:39:45,952 - JPype version is 1.6.0.
2025-12-08 15:39:45,953 - Starting Java virtual machine.
2025-12-08 15:39:47,748 - Java virtual machine has started.
2025-12-08 15:39:47,750 - Initializing stand-alone client.
2025-12-08 15:39:49,907 - Stand-alone client initialized.
2025-12-08 15:39:50,008 - Running on 1 processor core.


In [4]:
best_val = simulate_hydrodynamics(best_val, client)
client.clear()
print('best target =', -best_val)

2025-12-08 15:39:51,717 - Loading model "Prepared_NSS_0_10um_frog_RBC_2025.mph".
2025-12-08 15:39:59,714 - Finished loading model.
2025-12-08 15:40:01,131 - Running study "Study 1".
2025-12-08 15:40:07,251 - Finished solving study.
2025-12-08 15:40:07,343 - Evaluating vlct_1 on "Study 1/Solution 1" dataset.
2025-12-08 15:40:07,402 - Trying global evaluation.
2025-12-08 15:40:07,422 - Finished global evaluation.
2025-12-08 15:40:07,431 - Evaluating vlct_2 on "Study 1/Solution 1" dataset.
2025-12-08 15:40:07,477 - Trying global evaluation.
2025-12-08 15:40:07,491 - Finished global evaluation.
2025-12-08 15:40:07,501 - Evaluating vlct_3 on "Study 1/Solution 1" dataset.
2025-12-08 15:40:07,530 - Trying global evaluation.
2025-12-08 15:40:07,541 - Finished global evaluation.
2025-12-08 15:40:07,551 - Evaluating vlct_4 on "Study 1/Solution 1" dataset.
2025-12-08 15:40:07,577 - Trying global evaluation.
2025-12-08 15:40:07,586 - Finished global evaluation.
2025-12-08 15:40:07,597 - Evaluating

In [ ]:
best_structure = optimized_pop[0]

In [18]:
if not os.path.exists('logs_viz'):
    os.makedirs('logs_viz')

In [ ]:
visualiser = StructVizualizer(opt_params.domain) 
import matplotlib
matplotlib.use('Agg')
plt.figure(figsize=(7, 7))
info = {'fitness': best_structure.fitness,
        'type': 'Best Solution'}

visualiser.plot_structure(best_structure, info)
plt.savefig("best_structure1.png")
plt.show()

In [ ]:
import pickle
import mph
from setup_comsol import poly_add 

with open('best_structure.pkl', 'rb') as f:
    best = pickle.load(f)

client = mph.Client(cores=1)
model = client.load(r'Data\Prepared_NSS_0_10um_frog_RBC_2025.mph')
model = poly_add(model, best)
model.save('optimized_trap_best.mph')
client.clear()

2025-12-10 04:55:59,976 - JPype version is 1.6.0.
2025-12-10 04:55:59,979 - Starting Java virtual machine.
2025-12-10 04:56:06,008 - Java virtual machine has started.
2025-12-10 04:56:06,011 - Initializing stand-alone client.
2025-12-10 04:56:11,934 - Stand-alone client initialized.
2025-12-10 04:56:12,399 - Running on 1 processor core.
2025-12-10 04:56:12,407 - Loading model "EA part trac NSS_0 10um frog RBC_2025.mph".


In [ ]:
import os, glob, json, numpy as np
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "browser"

run_name = "run_name_2025-12-08_01_54_58"
run_dir = os.path.join("logs", run_name)

log_files = sorted(glob.glob(os.path.join(run_dir, "*.log")))
if not log_files:
    print("No .log files found in", run_dir)
else:
    gen_best = []
    for path in log_files:
        best = None
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                except json.JSONDecodeError:
                    continue
                fit = obj.get("fitness")
                if fit is None:
                    continue
                if isinstance(fit, list) and fit:
                    fit = fit[0]
                try:
                    fit = float(fit)
                except (TypeError, ValueError):
                    continue
                score = -fit
                if best is None or score > best:
                    best = score
        if best is not None:
            gen_best.append(best)

    if not gen_best:
        print("No fitness values parsed")
    else:
        gen_best = np.array(gen_best, dtype=float)
        best_running = np.maximum.accumulate(gen_best)
        x = list(range(len(best_running)))
        fig = go.Figure(go.Scatter(x=x, y=best_running, mode="lines+markers"))
        fig.update_layout(
            title=f"Convergence for {run_name}",
            xaxis_title="Iteration",
            yaxis_title="Best score",
        )
        fig.show()

        png_path = os.path.join(run_dir, "convergence.png")
        try:
            fig.write_image(png_path, width=800, height=600, scale=2)
            print(f"PNG saved to {png_path}")
        except Exception as e:
            print("Failed to save PNG (install 'kaleido' if missing):", e)

        print(f"Generations: {len(gen_best)}")
        print(f"Final best target: {best_running[-1]:.6f}")

PNG saved to logs\run_name_2025-12-08_01_54_58\convergence.png
Generations: 101
Final best target: 1.256862
